# All About LLMs: Positional Encoding & RoPE

This notebook builds up the theory of positional encoding in Transformers from first principles, then contrasts it with the Rotary Positional Embeddings (RoPE) used in modern LLMs (Llama, Mistral, Gemma, Qwen, Phi, etc.).

**Contents**
1. Why Transformers need positional information at all
2. Sinusoidal positional encoding — formula and design rationale
3. Key mathematical properties (relative position, uniqueness, boundedness, extrapolation)
4. Worked numerical example
5. Dot products between positional encodings
6. RoPE — motivation and formula
7. Worked numerical example for RoPE
8. Comparison: learned PE vs. sinusoidal PE vs. RoPE
9. Implementation detail: pairing convention (interleaved vs. rotate-half)
10. Extending RoPE beyond the training context length (PI, NTK-aware scaling, YaRN)
11. Code: verifying sinusoidal PE and RoPE numerically
12. Summary

## 1. Why Positional Encoding Is Needed

Self-attention is **permutation-invariant**: it computes weighted sums over all tokens regardless of their order. Without extra information, "the dog bit the man" and "the man bit the dog" would look identical to the attention mechanism, because the set of token embeddings is the same. Transformers therefore inject explicit position information into the input.

The original Transformer ("Attention Is All You Need") does this with a fixed **sinusoidal positional encoding** added to the token embeddings before the first layer.

## 2. Sinusoidal Positional Encoding — Formula

For position $pos$ and dimension index $i$ (with $d_{model}$ total embedding dimensions):

$$PE(pos, 2i) = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

Each pair of dimensions $(2i, 2i+1)$ oscillates at its own frequency, and the final input to the model is:

$$\text{Final Input}[pos] = \text{TokenEmbedding}[pos] + PE[pos]$$

### Why these frequencies?

The term $10000^{2i/d_{model}}$ controls the wavelength for each dimension pair:

- Small $i$ (low dimensions) → divisor ≈ 1 → **high frequency**, fast oscillation.
- Large $i$ (high dimensions) → divisor is large → **low frequency**, slow oscillation.

This produces a geometric progression of wavelengths, from about $2\pi$ up to $10000 \times 2\pi$. Different dimensions "tick" at different rates, giving the model positional signal at multiple scales simultaneously — fast pairs distinguish nearby positions, slow pairs distinguish coarse, long-range position.

## 3. Key Mathematical Properties

**a) Relative position via a linear (rotation) relationship — the most important property.**
Using the angle-addition identities
$$\sin(a+b) = \sin a\cos b + \cos a \sin b, \qquad \cos(a+b) = \cos a \cos b - \sin a \sin b$$
each 2D pair $(PE(pos,2i), PE(pos,2i+1))$ shifted by a fixed offset $k$ is exactly a **rotation**:

$$\begin{bmatrix} PE(pos+k, 2i) \\ PE(pos+k, 2i+1) \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} PE(pos, 2i) \\ PE(pos, 2i+1) \end{bmatrix}, \qquad \theta = \frac{k}{10000^{2i/d_{model}}}$$

Because this is a *linear* transformation, attention (which is built from linear projections and dot products) can learn to recover relative distance $k$ between tokens without needing to learn a complex nonlinear function.

**b) Periodicity + uniqueness.** Sine and cosine are individually periodic, but because every dimension pair oscillates at a different frequency, the combination of values across the full $d_{model}$-length vector is unique for each position within a very long range.

**c) Bounded values.** Every entry stays in $[-1, 1]$, matching the scale of normalized embeddings and helping training stability.

**d) Extrapolation.** Because $PE$ is a fixed closed-form function (not learned), it can be evaluated at positions longer than any sequence seen in training — unlike learned positional embedding tables, which have no representation for out-of-range positions.

## 4. Rotational angle:
$\theta_i$ (or $\phi_i$  is called the **rotation angle** or **rotary angle** for dimension-pair $i$ at a given position:

$$\theta_i(pos) = \frac{pos}{10000^{2i/d_{model}}}$$

It answers the question: *"by how many radians should this particular 2D pair of dimensions be rotated, given the token's position?"*
token's position?"


Rotation rotates 2D **pairs** of dimensions, not the whole vector at once. With `d_model = 6`, you have 6 dimensions total, which splits into exactly:

$$\frac{d_{model}}{2} = \frac{6}{2} = 3 \text{ pairs}$$

That's it — that's the whole reason there are 3 φᵢ values. Each φᵢ is the rotation angle for **one specific pair** of dimensions:

| Pair index $i$ | Dimensions it rotates | Rotation angle |
|---|---|---|
| $i=0$ | dims 0, 1 (Col0, Col1) | $\phi_0$ |
| $i=1$ | dims 2, 3 (Col2, Col3) | $\phi_1$ |
| $i=2$ | dims 4, 5 (Col4, Col5) | $\phi_2$ |

If `d_model` were 512 instead, you'd have 256 pairs, so $i$ would range from 0 to 255, and you'd need 256 different φᵢ values instead of 3. The number of φᵢ is always $d_{model}/2$ — it's a direct, mechanical consequence of "how many 2D pairs fit in this vector," nothing deeper than that.

#### Mathematical Breakdown
Here's the complete set for all three dimension-pairs (`i = 0, 1, 2`) with `d_model = 6`.

##### Step 1: The formula for each pair

$$\phi_i(pos) = \frac{pos}{10000^{2i/6}}$$

| Pair | Exponent $2i/6$ | Divisor $10000^{\text{exponent}}$ | Formula |
|---|---|---|---|
| $i=0$ | 0/6 = 0.000 | $10000^0 = 1$ | $\phi_0 = pos \div 1 = pos$ |
| $i=1$ | 2/6 = 0.333 | $10000^{0.333} = 100$ | $\phi_1 = pos \div 100 = 0.01\,pos$ |
| $i=2$ | 4/6 = 0.667 | $10000^{0.667} = 10000$ | $\phi_2 = pos \div 10000 = 0.0001\,pos$ |

##### Step 2: φ values at each position

| Word | pos | φ₀ ($pos\div1$) | φ₁ ($pos\div100$) | φ₂ ($pos\div10000$) |
|---|---|---|---|---|
| I | 0 | 0.00 | 0.0000 | 0.000000 |
| like | 1 | 1.00 | 0.0100 | 0.000100 |
| playing | 2 | 2.00 | 0.0200 | 0.000200 |
| football | 3 | 3.00 | 0.0300 | 0.000300 |
| in | 4 | 4.00 | 0.0400 | 0.000400 |
| my | 5 | 5.00 | 0.0500 | 0.000500 |
| leisure | 6 | 6.00 | 0.0600 | 0.000600 |
| time | 7 | 7.00 | 0.0700 | 0.000700 |

##### Step 3: sin/cos for each φ (→ table columns)

**φ₀ → Col0 (sin), Col1 (cos)**

| Word | pos | φ₀ | sin(φ₀) | cos(φ₀) |
|---|---|---|---|---|
| I | 0 | 0.00 | 0.0000 | 1.0000 |
| like | 1 | 1.00 | 0.8415 | 0.5403 |
| playing | 2 | 2.00 | 0.9093 | -0.4161 |
| football | 3 | 3.00 | 0.1411 | -0.9900 |
| in | 4 | 4.00 | -0.7568 | -0.6536 |
| my | 5 | 5.00 | -0.9589 | 0.2837 |
| leisure | 6 | 6.00 | -0.2794 | 0.9602 |
| time | 7 | 7.00 | 0.6570 | 0.7539 |

**φ₁ → Col2 (sin), Col3 (cos)**

| Word | pos | φ₁ | sin(φ₁) | cos(φ₁) |
|---|---|---|---|---|
| I | 0 | 0.0000 | 0.0000 | 1.00000 |
| like | 1 | 0.0100 | 0.0100 | 0.99995 |
| playing | 2 | 0.0200 | 0.0200 | 0.99980 |
| football | 3 | 0.0300 | 0.0300 | 0.99955 |
| in | 4 | 0.0400 | 0.0400 | 0.99920 |
| my | 5 | 0.0500 | 0.0500 | 0.99875 |
| leisure | 6 | 0.0600 | 0.0600 | 0.99820 |
| time | 7 | 0.0700 | 0.0699 | 0.99755 |

**φ₂ → Col4 (sin), Col5 (cos)**

| Word | pos | φ₂ | sin(φ₂) | cos(φ₂) |
|---|---|---|---|---|
| I | 0 | 0.000000 | 0.0000 | 1.0000 |
| like | 1 | 0.000100 | 0.0001 | 1.0000 |
| playing | 2 | 0.000200 | 0.0002 | 1.0000 |
| football | 3 | 0.000300 | 0.0003 | 1.0000 |
| in | 4 | 0.000400 | 0.0004 | 1.0000 |
| my | 5 | 0.000500 | 0.0005 | 1.0000 |
| leisure | 6 | 0.000600 | 0.0006 | 1.0000 |
| time | 7 | 0.000700 | 0.0007 | 1.0000 |

(φ₂ barely moves across 8 positions — this pair rotates ~10,000× slower than φ₀, so its sin/cos values look frozen at 4 decimal places. That's expected: it's the "slow" pair meant to capture long-range position information.)

##### Step 4: φ for rotation-by-k (position → position+k)

Same formulas, just plug in `k` instead of `pos`:

| From → To | k | φ₀ ($k\div1$) | φ₁ ($k\div100$) | φ₂ ($k\div10000$) |
|---|---|---|---|---|
| pos 1 → pos 3 | 2 | 2.00 | 0.0200 | 0.000200 |
| pos 0 → pos 1 | 1 | 1.00 | 0.0100 | 0.000100 |
| pos 2 → pos 5 | 3 | 3.00 | 0.0300 | 0.000300 |
| pos 3 → pos 7 | 4 | 4.00 | 0.0400 | 0.000400 |

All three pairs use the exact same `k`, but each pair's own divisor (1, 100, 10000) scales it down — which is the whole point of the multi-frequency design: φ₀ swings widely even for small `k`, while φ₂ barely moves, giving the model both fine and coarse relative-position resolution simultaneously.


##### Complete Table

| Word | pos | φ₀ (i=0) | φ₁ (i=1) | φ₂ (i=2) | Col0 (sin) | Col1 (cos) | Col2 (sin) | Col3 (cos) | Col4 (sin) | Col5 (cos) |
|------|-----|----------|----------|----------|------------|------------|------------|------------|------------|------------|
| I | 0 | 0.00 | 0.00 | 0.00 | 0.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 | 1.0000 |
| like | 1 | 1.00 | 0.01 | 0.0001 | 0.8415 | 0.5403 | 0.0100 | 0.99995 | 0.0001 | 1.0000 |
| playing | 2 | 2.00 | 0.02 | 0.0002 | 0.9093 | -0.4161 | 0.0200 | 0.99980 | 0.0002 | 1.0000 |
| football | 3 | 3.00 | 0.03 | 0.0003 | 0.1411 | -0.9900 | 0.0300 | 0.99955 | 0.0003 | 1.0000 |
| in | 4 | 4.00 | 0.04 | 0.0004 | -0.7568 | -0.6536 | 0.0400 | 0.99920 | 0.0004 | 1.0000 |
| my | 5 | 5.00 | 0.05 | 0.0005 | -0.9589 | 0.2837 | 0.0500 | 0.99875 | 0.0005 | 1.0000 |
| leisure | 6 | 6.00 | 0.06 | 0.0006 | -0.2794 | 0.9602 | 0.0600 | 0.99820 | 0.0006 | 1.0000 |
| time | 7 | 7.00 | 0.07 | 0.0007 | 0.6570 | 0.7539 | 0.0699 | 0.99755 | 0.0007 | 1.0000 |
#### Summary

- **3 φᵢ exist purely because $d_{model}/2 = 3$** — one φ per 2D pair of dimensions.
- **They're each different (not identical)** because using a spread of frequencies (fast → slow) lets the model resolve both short-range and long-range relative positions without ambiguity — a single fixed frequency couldn't do both.

## 4. Worked Numerical Example (Sinusoidal PE)

**Setup:** `d_model = 4` → 2 dimension pairs.
- Pair 1 ($i=0$): columns 0 (sin), 1 (cos) — divisor $10000^{0/4} = 1$, so $\theta_0(pos) = pos/1 = pos$ (**fast** frequency).
- Pair 2 ($i=1$): columns 2 (sin), 3 (cos) — divisor $10000^{2/4} = 100$, so $\theta_1(pos) = pos/100$ (**slow** frequency).

**Sentence:** "I like playing football in my leisure time"

### Step 1 — compute the angles $\theta_0$ and $\theta_1$ for each position

| Word | pos | $\theta_0 = pos/1$ | $\theta_1 = pos/100$ |
|------|-----|---------------------|------------------------|
| I | 0 | 0.00 | 0.0000 |
| like | 1 | 1.00 | 0.0100 |
| playing | 2 | 2.00 | 0.0200 |
| football | 3 | 3.00 | 0.0300 |
| in | 4 | 4.00 | 0.0400 |
| my | 5 | 5.00 | 0.0500 |
| leisure | 6 | 6.00 | 0.0600 |
| time | 7 | 7.00 | 0.0700 |

### Step 2 — take sin/cos of each angle to get the PE vector

| Word | pos | Col0 sin($\theta_0$) | Col1 cos($\theta_0$) | Col2 sin($\theta_1$) | Col3 cos($\theta_1$) |
|------|-----|------------------------|------------------------|------------------------|------------------------|
| I | 0 | 0.0000 | 1.0000 | 0.0000 | 1.0000 |
| like | 1 | 0.8415 | 0.5403 | 0.0100 | 0.99995 |
| playing | 2 | 0.9093 | -0.4161 | 0.0200 | 0.99980 |
| football | 3 | 0.1411 | -0.9900 | 0.0300 | 0.99955 |
| in | 4 | -0.7568 | -0.6536 | 0.0400 | 0.99920 |
| my | 5 | -0.9589 | 0.2837 | 0.0500 | 0.99875 |
| leisure | 6 | -0.2794 | 0.9602 | 0.0600 | 0.99820 |
| time | 7 | 0.6570 | 0.7539 | 0.0699 | 0.99755 |

### Adding to token embeddings (dummy example)

| Word | Dummy Token Embedding | Positional Encoding | **Final Input** |
|------|------------------------|----------------------|-------------------|
| I | [0.20, -0.50, 0.40, 0.10] | [0.0000, 1.0000, 0.0000, 1.0000] | [0.2000, 0.5000, 0.4000, 1.1000] |
| like | [0.10, 0.60, -0.30, 0.80] | [0.8415, 0.5403, 0.0100, 0.99995] | [0.9415, 1.1403, -0.2900, 1.79995] |
| playing | [0.50, -0.30, 0.80, 0.20] | [0.9093, -0.4161, 0.0200, 0.99980] | [1.4093, -0.7161, 0.8200, 1.19980] |
| football | [0.30, 0.40, -0.20, 0.60] | [0.1411, -0.9900, 0.0300, 0.99955] | [0.4411, -0.5900, -0.1700, 1.59955] |

### Verifying the rotation property

Take "like" ($pos=1$) → "football" ($pos=3$), so $k=2$. For the fast pair ($i=0$), the shift angle is

$$\theta = \frac{k}{10000^{2i/d_{model}}} = \frac{2}{10000^{0/4}} = \frac{2}{1} = 2 \text{ radians}$$

and the pair is stored in $(x,y) = (\cos(pos), \sin(pos))$ order. Shifting the angle by $\theta=2$ radians should reproduce $(\cos(3), \sin(3)) = (-0.9900, 0.1411)$. Starting from $(\cos(1), \sin(1)) = (0.5403, 0.8415)$ and applying the rotation matrix:

$$\begin{bmatrix}x'\\y'\end{bmatrix} = \begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}\begin{bmatrix}x\\y\end{bmatrix}, \qquad \cos(2)\approx-0.4161,\ \sin(2)\approx0.9093$$

$$x' = \cos(2)(0.5403) - \sin(2)(0.8415) = (-0.4161)(0.5403) - (0.9093)(0.8415) = -0.2248 - 0.7652 = -0.9900 = \cos(3)\ \checkmark$$
$$y' = \sin(2)(0.5403) + \cos(2)(0.8415) = (0.9093)(0.5403) + (-0.4161)(0.8415) = 0.4913 - 0.3501 = 0.1412 \approx \sin(3)\ \checkmark$$

For the slow pair ($i=1$), the same offset $k=2$ gives a much smaller shift angle:

$$\theta = \frac{k}{10000^{2i/d_{model}}} = \frac{2}{10000^{2/4}} = \frac{2}{100} = 0.02 \text{ radians}$$

confirming that low dimensions rotate fast per step while high dimensions barely rotate — the geometric progression of frequencies described in Section 2.

The rotation exactly reproduces $PE_{(3,\cdot)}$ from $PE_{(1,\cdot)}$, confirming that a fixed offset $k$ corresponds to a fixed rotation angle, independent of the starting position.

## 5. Dot Products Between Positional Encodings

Attention scores involve dot products of (embedding + PE) vectors. The PE component of that dot product depends only on relative distance $k$, which is what lets the model use position information without extra learned parameters.

| From → To | k | Dot product of PE vectors | Interpretation |
|-----------|---|---------------------------|-----------------|
| I → I | 0 | 2.0000 | maximum (identical position) |
| I → like | 1 | 1.5403 | high similarity |
| I → playing | 2 | 0.5837 | moderate |
| like → leisure | 5 | 1.2825 | still fairly high — periodicity means similarity doesn't decay monotonically with distance |

Example calculation (I → like, $k=1$): $PE_0=[0,1,0,1]$, $PE_1=[0.8415, 0.5403, 0.0100, 0.99995]$

$$\text{dot} = 0\times0.8415 + 1\times0.5403 + 0\times0.0100 + 1\times0.99995 = 1.54025$$

**Limitation to note:** because sin/cos are periodic, the dot product is not a monotonic function of $k$ — very distant positions can occasionally look similar again. This non-monotonic decay, plus the fact that positional information is only *added* (and can be partially washed out by deep stacks of layers), is part of the motivation for RoPE, covered next.

In [7]:
## Absolute positional Encoding
import numpy as np

def sinusoidal_pe(seq_len, d_model, base=10000.0):
    """PE[pos, 2i] = sin(pos / base^(2i/d_model)); PE[pos, 2i+1] = cos(...)"""
    pos = np.arange(seq_len)[:, None]                      # (seq_len, 1)
    i = np.arange(d_model // 2)[None, :]                    # (1, d_model/2)
    angles = pos / base ** (2 * i / d_model)                 # (seq_len, d_model/2)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

words = ["I", "like", "playing", "football", "in", "my", "leisure", "time"]
pe = sinusoidal_pe(seq_len=len(words), d_model=4)
for w, row in zip(words, pe):
    print(f"{w:10s} pos={words.index(w)}  PE={np.round(row, 4)}")


I          pos=0  PE=[0. 1. 0. 1.]
like       pos=1  PE=[0.8415 0.5403 0.01   1.    ]
playing    pos=2  PE=[ 0.9093 -0.4161  0.02    0.9998]
football   pos=3  PE=[ 0.1411 -0.99    0.03    0.9996]
in         pos=4  PE=[-0.7568 -0.6536  0.04    0.9992]
my         pos=5  PE=[-0.9589  0.2837  0.05    0.9988]
leisure    pos=6  PE=[-0.2794  0.9602  0.06    0.9982]
time       pos=7  PE=[0.657  0.7539 0.0699 0.9976]


In [8]:
# Dot-product dry run: reproduce every row of the Section 5 table by hand-style computation
pairs = [
    ("I", "I"),
    ("I", "like"),
    ("I", "playing"),
    ("like", "leisure"),
]

for a, b in pairs:
    pos_a, pos_b = words.index(a), words.index(b)
    pe_a, pe_b = pe[pos_a], pe[pos_b]
    k = pos_b - pos_a
    terms = pe_a * pe_b
    dot = terms.sum()
    term_str = " + ".join(f"{x:.4f}*{y:.4f}" for x, y in zip(pe_a, pe_b))
    print(f"{a:8s} -> {b:8s}  k={k}")
    print(f"  PE({a})={np.round(pe_a,4)}  PE({b})={np.round(pe_b,4)}")
    print(f"  dot = {term_str} = {dot:.4f}\n")


I        -> I         k=0
  PE(I)=[0. 1. 0. 1.]  PE(I)=[0. 1. 0. 1.]
  dot = 0.0000*0.0000 + 1.0000*1.0000 + 0.0000*0.0000 + 1.0000*1.0000 = 2.0000

I        -> like      k=1
  PE(I)=[0. 1. 0. 1.]  PE(like)=[0.8415 0.5403 0.01   1.    ]
  dot = 0.0000*0.8415 + 1.0000*0.5403 + 0.0000*0.0100 + 1.0000*1.0000 = 1.5403

I        -> playing   k=2
  PE(I)=[0. 1. 0. 1.]  PE(playing)=[ 0.9093 -0.4161  0.02    0.9998]
  dot = 0.0000*0.9093 + 1.0000*-0.4161 + 0.0000*0.0200 + 1.0000*0.9998 = 0.5837

like     -> leisure   k=5
  PE(like)=[0.8415 0.5403 0.01   1.    ]  PE(leisure)=[-0.2794  0.9602  0.06    0.9982]
  dot = 0.8415*-0.2794 + 0.5403*0.9602 + 0.0100*0.0600 + 1.0000*0.9982 = 1.2824



## 6. RoPE — Rotary Positional Embeddings

### Quick refresher: radians

A radian is the unit of rotation angle on the unit circle (radius = 1). $\text{degrees} = \text{radians}\times\frac{180}{\pi}$. Landmarks: $0$ rad → $(1,0)$; $\pi/2\approx1.57$ rad → $(0,1)$ (90°); $\pi\approx3.14$ rad → $(-1,0)$ (180°); $2\pi\approx6.28$ rad → back to $(1,0)$ (360°).

### Motivation

Sinusoidal PE **adds** a position vector to the token embedding once, at the input layer. RoPE — used in Llama, Mistral, Gemma, Qwen, Phi, and most modern LLMs — takes a different approach: instead of adding anything, it **rotates** the Query and Key vectors inside every attention layer by an angle that depends on token position.

Advantages over additive sinusoidal PE:
- Relative position is encoded **explicitly** in the attention dot product, not just recoverable in principle.
- Better extrapolation to sequence lengths beyond training.
- No positional signal is mixed into the value/residual stream — it lives purely in how Q and K interact.

### Formula

For dimension-pair index $m$ and position $pos$, with $d_{model}$ total dimensions:

$$\theta_m(pos) = \frac{pos}{10000^{2m/d_{model}}}$$

For a vector $x = [x_0, x_1, x_2, x_3, \dots]$, grouped into pairs $(x_{2m}, x_{2m+1})$, RoPE rotates each pair:

$$\begin{bmatrix} x'_{2m} \\ x'_{2m+1} \end{bmatrix} = \begin{bmatrix} \cos\theta_m & -\sin\theta_m \\ \sin\theta_m & \cos\theta_m \end{bmatrix} \begin{bmatrix} x_{2m} \\ x_{2m+1} \end{bmatrix}$$

This rotation is applied to the **Query** and **Key** projections (not the Value, and not the raw token embedding) inside every attention layer.

### Why the attention score depends only on relative position

The attention score between a rotated query at position $i$ and a rotated key at position $j$ is:

$$q_i^\top k_j = \left(R(\theta_i)\, W_q x_i\right)^\top \left(R(\theta_j)\, W_k x_j\right) = (W_q x_i)^\top R(\theta_i)^\top R(\theta_j) (W_k x_j)$$

Because rotation matrices compose ($R(\theta_i)^\top R(\theta_j) = R(\theta_j - \theta_i)$), this depends only on $\theta_j - \theta_i$, i.e. only on the **relative distance** $k = j - i$ — never on the absolute positions $i$ or $j$ individually. This is a strictly stronger relative-position guarantee than sinusoidal PE provides.

Let's unpack each phrase in that sentence individually, since it's dense with linear algebra vocabulary.

#### 1. "Rotation matrices compose"

This means: if you multiply two rotation matrices together, you get another rotation matrix, and the resulting angle is just the **sum** of the two angles.

$$R(a) \cdot R(b) = R(a+b)$$

Intuitively: rotating something by angle $a$, then rotating the result by angle $b$, is the same as rotating it once by $a+b$. This is a basic geometric fact — rotations "stack" additively in their angles. Formally, this is the **group property** of $SO(2)$ (the set of 2D rotation matrices): they form a group under multiplication, and that group is isomorphic to addition of angles.

#### 2. Why $R(\theta_i)^\top = R(-\theta_i)$

This is the key algebraic trick the derivation relies on. A rotation matrix is:

$$R(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Its transpose swaps the off-diagonal entries:

$$R(\theta)^\top = \begin{bmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{bmatrix}$$

Compare that to $R(-\theta)$, using $\cos(-\theta)=\cos\theta$ and $\sin(-\theta)=-\sin\theta$:

$$R(-\theta) = \begin{bmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{bmatrix}$$

Identical. So **transposing a rotation matrix is the same as negating its angle**. This holds for any orthogonal matrix ($R^\top = R^{-1}$), and rotation matrices are orthogonal because rotation preserves length and angles — it never stretches or skews.

#### 3. Combining both facts to get $R(\theta_i)^\top R(\theta_j) = R(\theta_j - \theta_i)$

Now just chain the two facts together:

$$R(\theta_i)^\top R(\theta_j) = R(-\theta_i)\cdot R(\theta_j) = R(-\theta_i + \theta_j) = R(\theta_j - \theta_i)$$

Step 1 uses fact #2 (transpose = negate angle). Step 2 uses fact #1 (rotations compose by adding angles). That's the entire derivation — two well-known facts about rotation matrices chained together.

#### 4. What "relative distance $k = j - i$" means

Recall $\theta_i = i \cdot \omega$ and $\theta_j = j \cdot \omega$ for a fixed frequency $\omega$ (what the doc called $10000^{-2m/d}$). So:

$$\theta_j - \theta_i = (j - i)\cdot \omega$$

The angle that survives in the final rotation matrix is proportional to $j - i$ — the **gap between the two token positions**, commonly denoted $k$. So $k=5$ means "key is 5 tokens after the query," and it doesn't matter if that's positions $(2,7)$, $(10,15)$, or $(100,105)$ — the resulting rotation angle $k\omega$ is identical in all three cases.

#### 5. Why this means it "never depends on absolute positions individually"

This is the concrete, checkable claim. Take two different pairs of positions with the *same* offset:

- $i=2, j=7 \Rightarrow \theta_j - \theta_i = (7-2)\omega = 5\omega$
- $i=100, j=105 \Rightarrow \theta_j - \theta_i = (105-100)\omega = 5\omega$

Both give the **exact same rotation matrix** $R(5\omega)$ applied inside the dot product, even though the absolute positions are completely different. The attention score $q_i^\top k_j$ literally cannot distinguish "token 7 attending to token 2" from "token 105 attending to token 100" — only the 5-token gap matters. That's what "never on the absolute positions individually" is asserting, and it's provable, not approximate.

#### 6. Why this is "strictly stronger" than what sinusoidal PE provides

This is the comparison worth being precise about, since sinusoidal PE is *also* often described as giving "relative position" — but in a weaker sense.

**Sinusoidal PE's relative property:** The original Transformer paper showed that $p_{i+k}$ can be expressed as a **linear function** of $p_i$ (a rotation-like transformation exists at the level of the position vectors themselves). This means relative position is theoretically *recoverable* — a model *could* learn to extract it.

**But recall the four-term expansion from earlier:**

$$q_i^\top k_j = t_i^\top W_q^\top W_k t_j + t_i^\top W_q^\top W_k p_j + p_i^\top W_q^\top W_k t_j + p_i^\top W_q^\top W_k p_j$$

Even if $p_i, p_j$ individually have a nice relative structure, this sum of four separate terms does **not** algebraically collapse into a clean function of $(j-i)$ alone. The token-content terms and position terms are tangled together by addition, and the network has to *learn* weights $W_q, W_k$ that approximately isolate the relative signal — with no guarantee it does so perfectly, and no guarantee it generalizes to position gaps unseen during training.

**RoPE's relative property**, by contrast, is **exact and structural**: the derivation above shows the *entire* dot product — one single term, not four — collapses to a function of $(j-i)$ **for any** $W_q, W_k$, before any learning happens. It's baked into the geometry of rotation, not something the model has to discover through training.

That's the precise meaning of "strictly stronger": sinusoidal PE makes relative position *learnable*; RoPE makes relative position *mathematically guaranteed*, independent of what the learned weight matrices turn out to be.

### 6.1 Worked Numerical Example (RoPE)

**Setup:** same `d_model = 4` toy embeddings as before, applied to the Query/Key vector for each word (here just illustrated on the raw dummy embedding, standing in for a Query or Key vector).

- Fast pair ($m=0$): $\theta_0(pos) = pos$
- Slow pair ($m=1$): $\theta_1(pos) = pos/100$

**Rotating "football"'s vector (pos = 3), dummy vector $[0.30, 0.40, -0.20, 0.60]$**

Fast pair, $\theta_0 = 3$ rad, $\cos(3)\approx-0.9900$, $\sin(3)\approx0.1411$:

$$x'_0 = \cos(3)(0.30) - \sin(3)(0.40) = -0.2970 - 0.0564 = -0.3534$$
$$x'_1 = \sin(3)(0.30) + \cos(3)(0.40) = 0.0423 - 0.3960 = -0.3537$$

Slow pair, $\theta_1 = 0.03$ rad, $\cos(0.03)\approx0.99955$, $\sin(0.03)\approx0.030$:

$$x'_2 = \cos(0.03)(-0.20) - \sin(0.03)(0.60) = -0.19991 - 0.0180 = -0.2179$$
$$x'_3 = \sin(0.03)(-0.20) + \cos(0.03)(0.60) = -0.0060 + 0.5997 = 0.5937$$

**Rotated vector:** $[-0.3534, -0.3537, -0.2179, 0.5937]$

### Relative-distance check: "like" (pos=1) → "football" (pos=3)

Fast-pair relative angle: $\theta_0(3) - \theta_0(1) = 3 - 1 = 2$ rad. Slow-pair relative angle: $\theta_1(3)-\theta_1(1) = 0.03-0.01 = 0.02$ rad. Exactly the same offsets used for the sinusoidal-PE rotation example in section 4 — because both schemes use the same $10000^{2i/d_{model}}$ frequency base, only *where* the rotation is applied differs (added once to the embedding vs. applied to Q/K every layer).

### 6.3 Comparison: Learned PE vs. Sinusoidal PE vs. RoPE

| Aspect | Learned Absolute PE | Sinusoidal PE | RoPE |
|--------|----------------------|----------------|------|
| Where applied | Added to token embedding, once, at input (looked up from a trained table) | Added to token embedding, once, at input | Applied to Query & Key, inside every attention layer |
| Relative position | Not explicit — model must infer it from learned vectors | Indirect — recoverable via dot product | Direct — provably depends only on $i-j$ |
| Length extrapolation | None — no embedding exists for positions beyond the trained table size | Moderate | Strong (and further extendable — see Section 10) |
| Learned parameters | Full $seq\_len \times d_{model}$ embedding table | None (fixed function) | None (fixed function) |
| Used in | GPT-2, BERT, early GPT-family models | Original Transformer | Llama, Mistral, Gemma, Qwen, Phi, most current LLMs |

**Learned absolute positional embeddings**, used by GPT-2 and BERT, replace the fixed sin/cos formula with an ordinary trainable embedding table indexed by position (just like a token embedding table, but indexed by slot instead of vocabulary id). This lets the model shape its own positional geometry rather than being constrained to sinusoids, but it has a hard failure mode: position $p$ has **no representation at all** if $p \geq$ the maximum sequence length seen during training, so these models cannot extrapolate to longer contexts without further fine-tuning or interpolation tricks.

### 6.4 RoPE Example-2
 Positional Encoding Matrix for the Sentence

**Sentence:** "I like playing football in my leisure time"

Rotation angle φ is calculated **relative to position 0** (φ = pos / denom).

Positional Encoding Table for `d_model = 6`

| Word | pos | φ₀ (i=0) | φ₁ (i=1) | φ₂ (i=2) | Col0 (sin) | Col1 (cos) | Col2 (sin) | Col3 (cos) | Col4 (sin) | Col5 (cos) |
|------|-----|----------|----------|----------|------------|------------|------------|------------|------------|------------|
| I | 0 | 0.00 | 0.00 | 0.00 | 0.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 | 1.0000 |
| like | 1 | 1.00 | 0.01 | 0.0001 | 0.8415 | 0.5403 | 0.0100 | 0.99995 | 0.0001 | 1.0000 |
| playing | 2 | 2.00 | 0.02 | 0.0002 | 0.9093 | -0.4161 | 0.0200 | 0.99980 | 0.0002 | 1.0000 |
| football | 3 | 3.00 | 0.03 | 0.0003 | 0.1411 | -0.9900 | 0.0300 | 0.99955 | 0.0003 | 1.0000 |
| in | 4 | 4.00 | 0.04 | 0.0004 | -0.7568 | -0.6536 | 0.0400 | 0.99920 | 0.0004 | 1.0000 |
| my | 5 | 5.00 | 0.05 | 0.0005 | -0.9589 | 0.2837 | 0.0500 | 0.99875 | 0.0005 | 1.0000 |
| leisure | 6 | 6.00 | 0.06 | 0.0006 | -0.2794 | 0.9602 | 0.0600 | 0.99820 | 0.0006 | 1.0000 |
| time | 7 | 7.00 | 0.07 | 0.0007 | 0.6570 | 0.7539 | 0.0699 | 0.99755 | 0.0007 | 1.0000 |

#### 3. Rotation Property & Relative Distance (k)

**Key Idea**: Moving from position `pos` to `pos + k` is equivalent to rotating each 2D pair by angle φ.

#### Rotation Formula

For each pair `i`:

$$
\phi = \frac{k}{10000^{2i / d_{model}}}
$$

Each pair is stored as `(sin(pos·w), cos(pos·w))`, i.e. **(sin, cos)** order, not (cos, sin). Using the angle-addition identities:

$$
\sin(pos+k) = \sin(pos)\cos(k) + \cos(pos)\sin(k)
$$
$$
\cos(pos+k) = \cos(pos)\cos(k) - \sin(pos)\sin(k)
$$

this corresponds to the rotation matrix:

$$
R(\phi) =
\begin{bmatrix}
\cos\phi & \sin\phi \\
-\sin\phi & \cos\phi
\end{bmatrix}
$$

Then:
$$
\begin{bmatrix}
PE(pos+k, 2i) \\
PE(pos+k, 2i+1)
\end{bmatrix}
= R(\phi) \times
\begin{bmatrix}
PE(pos, 2i) \\
PE(pos, 2i+1)
\end{bmatrix}
$$

i.e.
$$
\text{new\_sin} = \cos\phi \times \text{old\_sin} + \sin\phi \times \text{old\_cos}
$$
$$
\text{new\_cos} = -\sin\phi \times \text{old\_sin} + \cos\phi \times \text{old\_cos}
$$

---
**Goal**: Show that $PE(pos + k)$ can be obtained by rotating $PE(pos)$ using a rotation matrix.

#### Example: From "like" (pos=1) to "football" (pos=3), k=2

**Fast Pair (i=0)**

- Old vector (pos=1): `[0.8415, 0.5403]`
- Rotation angle: $\phi = k / 1 = 2.0$ radians
- $\cos\phi = -0.4161$, $\sin\phi = 0.9093$

##### Step-by-Step Matrix Multiplication

**New sin (Column 0):**
$$
\text{new\_sin} = (\cos\phi \times \text{old\_sin}) + (\sin\phi \times \text{old\_cos})
$$
$$
= (-0.4161 \times 0.8415) + (0.9093 \times 0.5403)
$$
$$
= -0.3501 + 0.4913 = 0.1412
$$

**New cos (Column 1):**
$$
\text{new\_cos} = (-\sin\phi \times \text{old\_sin}) + (\cos\phi \times \text{old\_cos})
$$
$$
= (-0.9093 \times 0.8415) + (-0.4161 \times 0.5403)
$$
$$
= -0.7652 - 0.2248 = -0.9900
$$

**Check**: this matches the table row for "football" (pos=3): `[0.1411, -0.9900]`. The rotated vector equals the direct sin/cos formula exactly — no hand-waving needed.

---

**From pos=0 to pos=1 (k=1), Fast Pair**

- Old: `[0.0000, 1.0000]`
- φ = 1.0, $\cos(1) = 0.5403$, $\sin(1) = 0.8415$

- new_sin = (0.5403 × 0) + (0.8415 × 1) = **0.8415**
- new_cos = (-0.8415 × 0) + (0.5403 × 1) = **0.5403**

**Actual at pos=1**: `[0.8415, 0.5403]` — exact match.

---

**Slow Pair (i=1)**: φ = 2/100 = **0.02 radians** (very small rotation)


#### 5. Dot Product Examples (Position-Position)
**Why do we need this?**
The dot product between two positional encodings is what the attention mechanism uses to understand **relative distance** between tokens. Higher dot product = more similar positions = model naturally pays more attention to nearby tokens.

### Detailed Calculations

#### Example 1: "I" (pos=0) vs "I" (pos=0), k=0

**PE0** = [0.0000, 1.0000, 0.0000, 1.0000]

$$
\text{dot} = (0.0000 \times 0.0000) + (1.0000 \times 1.0000) + (0.0000 \times 0.0000) + (1.0000 \times 1.0000) = 2.0000
$$

#### Example 2: "I" (pos=0) vs "like" (pos=1), k=1

**PE1** = [0.8415, 0.5403, 0.0100, 0.99995]

$$
\begin{align*}
\text{dot} &= (0.0000 \times 0.8415) \\
&\quad + (1.0000 \times 0.5403) \\
&\quad + (0.0000 \times 0.0100) \\
&\quad + (1.0000 \times 0.99995) \\
&= 0 + 0.5403 + 0 + 0.99995 = \mathbf{1.54025}
\end{align*}
$$

#### Example 3: "I" (pos=0) vs "playing" (pos=2), k=2

**PE2** = [0.9093, -0.4161, 0.0200, 0.99980]

$$
\begin{align*}
\text{dot} &= (0.0000 \times 0.9093) \\
&\quad + (1.0000 \times -0.4161) \\
&\quad + (0.0000 \times 0.0200) \\
&\quad + (1.0000 \times 0.99980) \\
&= 0 - 0.4161 + 0 + 0.99980 = \mathbf{0.5837}
\end{align*}
$$

#### Example 4: "like" (pos=1) vs "leisure" (pos=6), k=5

**PE1** = [0.8415, 0.5403, 0.0100, 0.99995]
**PE6** = [-0.2794, 0.9602, 0.0600, 0.99820]

$$
\begin{align*}
\text{dot} &= (0.8415 \times -0.2794) \\
&\quad + (0.5403 \times 0.9602) \\
&\quad + (0.0100 \times 0.0600) \\
&\quad + (0.99995 \times 0.99820) \\
&= -0.2351 + 0.5188 + 0.0006 + 0.99815 = \mathbf{1.28245}
\end{align*}
$$

---



---
## Summary of Dot Products

| From → To       | k   | Dot Product | Interpretation                  |
|-----------------|-----|-------------|---------------------------------|
| I → I           | 0   | 2.0000      | Maximum (perfect match)         |
| I → like        | 1   | 1.5403      | High similarity                 |
| I → playing     | 2   | 0.5837      | Medium                          |
| like → leisure  | 5   | 1.2825      | Still reasonably high           |

**Why we need dot products**: They allow the attention mechanism to naturally weigh closer tokens higher without any extra parameters. This is one of the main reasons the sinusoidal positional encoding works so well.

## 9. Implementation Detail: Pairing Convention (Interleaved vs. "Rotate-Half")

Section 6 describes RoPE pairing dimensions **adjacently**: $(x_0,x_1), (x_2,x_3), \dots, (x_{d-2}, x_{d-1})$ — this is the interleaved convention used in the original RoPE paper and is mathematically the cleanest way to present the idea.

Most production implementations (Llama, GPT-NeoX, Hugging Face `transformers`) instead use a **"rotate-half"** convention that pairs each early dimension with its counterpart halfway across the vector:

$$(x_0, x_{d/2}), (x_1, x_{d/2+1}), \dots, (x_{d/2-1}, x_{d-1})$$

Concretely, splitting $x$ into two halves $x_{1:d/2}$ and $x_{d/2+1:d}$, the rotation is applied as:

$$x' = x \odot \cos(\theta) + \text{rotate\_half}(x) \odot \sin(\theta), \qquad \text{rotate\_half}(x) = [-x_{d/2+1:d},\ x_{1:d/2}]$$

where $\cos(\theta)$ and $\sin(\theta)$ are each broadcast/repeated across both halves (i.e. $\theta_0,\dots,\theta_{d/2-1}$ used twice). This produces the *same set* of 2D rotations — each frequency $\theta_m$ still rotates exactly one pair of coordinates by the same angle — just with the pair members reshuffled to non-adjacent positions in memory. It's purely a layout choice that makes the operation faster on GPUs (it becomes two element-wise multiplies and a concat/slice instead of a strided gather), and it does **not** change any of the mathematical properties derived in Section 6 (relative-position dependence, norm preservation, etc.). It does mean that if you mix code using one convention with weights trained under the other, the model will silently produce wrong results — a real, easy-to-hit bug in practice.

## 10. Extending RoPE Beyond the Training Context Length

RoPE extrapolates better than sinusoidal PE out of the box, but attention quality still degrades once inference sequences run far past the training length — the model simply never saw those rotation angles during training. Several training-free or cheap-fine-tune techniques address this by rescaling how position maps to angle:

- **Linear (position) interpolation (PI).** Compress positions before computing angles: use $\theta_m(pos) = \frac{pos/s}{10000^{2m/d_{model}}}$ for a scale factor $s = L_{new}/L_{train}$. This squeezes a longer sequence into the range of angles the model already learned, at the cost of reduced resolution between adjacent positions (short-range relationships get blurrier).
- **NTK-aware scaling.** Instead of scaling every position uniformly, rescale the frequency *base* (the $10000$) upward: $\text{base}' = \text{base}\cdot s^{d_{model}/(d_{model}-2)}$. This stretches the slow (high-$i$) dimensions — which carry long-range/coarse position information — much more than the fast dimensions, so short-range resolution (which the fast dimensions provide) is preserved while long-range capacity is extended. Often works well with no fine-tuning at all.
- **YaRN (Yet another RoPE extensioN).** Combines NTK-style frequency rescaling with an attention-temperature adjustment, treating each frequency dimension differently based on its wavelength relative to the target context length (interpolating low frequencies, leaving high frequencies mostly alone, blending in between). It is the technique behind many long-context Llama/Mistral variants and typically needs only a short fine-tuning run to reach strong quality at several times the original training length.

The common thread: because RoPE's positional signal is just an angle $\theta_m(pos)$ computed from a formula (not a fixed lookup table), that formula can be *edited post-hoc* — this is precisely the flexibility that learned absolute positional embeddings (Section 8) don't have, since their positions beyond the training table simply don't exist.

## 11. Code: Verifying Sinusoidal PE and RoPE Numerically

The cells below implement both schemes in plain NumPy and reproduce the worked-by-hand numbers from Sections 4 and 7, then verify the two core theoretical claims: (1) a fixed positional offset $k$ is a fixed rotation for sinusoidal PE, and (2) the RoPE attention score between positions $i$ and $j$ depends only on $j-i$.

In [9]:
import numpy as np

def sinusoidal_pe(seq_len, d_model, base=10000.0):
    """PE[pos, 2i] = sin(pos / base^(2i/d_model)); PE[pos, 2i+1] = cos(...)"""
    pos = np.arange(seq_len)[:, None]                      # (seq_len, 1)
    i = np.arange(d_model // 2)[None, :]                    # (1, d_model/2)
    angles = pos / base ** (2 * i / d_model)                 # (seq_len, d_model/2)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles)
    return pe

words = ["I", "like", "playing", "football", "in", "my", "leisure", "time"]
pe = sinusoidal_pe(seq_len=len(words), d_model=4)
for w, row in zip(words, pe):
    print(f"{w:10s} pos={words.index(w)}  PE={np.round(row, 4)}")


I          pos=0  PE=[0. 1. 0. 1.]
like       pos=1  PE=[0.8415 0.5403 0.01   1.    ]
playing    pos=2  PE=[ 0.9093 -0.4161  0.02    0.9998]
football   pos=3  PE=[ 0.1411 -0.99    0.03    0.9996]
in         pos=4  PE=[-0.7568 -0.6536  0.04    0.9992]
my         pos=5  PE=[-0.9589  0.2837  0.05    0.9988]
leisure    pos=6  PE=[-0.2794  0.9602  0.06    0.9982]
time       pos=7  PE=[0.657  0.7539 0.0699 0.9976]


In [10]:
# Verify: shifting position by k is exactly a 2D rotation of each (2i, 2i+1) pair
def rot2d(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])

d_model, base = 4, 10000.0
pos_from, pos_to = 1, 3        # "like" -> "football", k=2
k = pos_to - pos_from

pe_full = sinusoidal_pe(seq_len=8, d_model=d_model, base=base)
for i in range(d_model // 2):
    theta = k / base ** (2 * i / d_model)
    pair_from = pe_full[pos_from, 2*i:2*i+2]      # stored as (sin, cos)
    xy_from = pair_from[::-1]                      # reorder to (cos, sin) = (x, y)
    xy_rotated = rot2d(theta) @ xy_from
    xy_true = pe_full[pos_to, 2*i:2*i+2][::-1]
    print(f"pair i={i}: theta={theta:.4f}  rotated={np.round(xy_rotated,4)}  "
          f"true={np.round(xy_true,4)}  match={np.allclose(xy_rotated, xy_true, atol=1e-3)}")


pair i=0: theta=2.0000  rotated=[-0.99    0.1411]  true=[-0.99    0.1411]  match=True
pair i=1: theta=0.0200  rotated=[0.9996 0.03  ]  true=[0.9996 0.03  ]  match=True


In [11]:
# RoPE: rotate a Q/K vector at a given position (interleaved convention, matches Section 6/7)
def rope_rotate(x, pos, base=10000.0):
    d = len(x)
    m = np.arange(d // 2)
    theta = pos / base ** (2 * m / d)
    x_pairs = x.reshape(-1, 2)                 # (d/2, 2) -> [(x0,x1), (x2,x3), ...]
    c, s = np.cos(theta), np.sin(theta)
    x0, x1 = x_pairs[:, 0], x_pairs[:, 1]
    rotated = np.stack([c * x0 - s * x1, s * x0 + c * x1], axis=1)
    return rotated.reshape(-1)

x_football = np.array([0.30, 0.40, -0.20, 0.60])
rotated = rope_rotate(x_football, pos=3)
print("rotated 'football' vector:", np.round(rotated, 4))
# expect approx [-0.3534, -0.3537, -0.2179, 0.5937], matching the hand-worked Section 7 result

# Verify: attention score q_i . k_j depends only on relative distance k = j - i
def rope_dot(x_i, i, x_j, j, base=10000.0):
    return rope_rotate(x_i, i, base) @ rope_rotate(x_j, j, base)

q = np.array([0.10, 0.60, -0.30, 0.80])   # stand-in Query vector
k = np.array([0.30, 0.40, -0.20, 0.60])   # stand-in Key vector

for (i, j) in [(1, 3), (2, 4), (5, 7)]:   # all have relative distance 2
    print(f"i={i}, j={j}, j-i={j-i}: score={rope_dot(q, i, k, j):.6f}")


rotated 'football' vector: [-0.3534 -0.3537 -0.2179  0.5937]
i=1, j=3, j-i=2: score=0.555234
i=2, j=4, j-i=2: score=0.555234
i=5, j=7, j-i=2: score=0.555234


In [12]:
# "Rotate-half" convention (Llama/GPT-NeoX/HF style) -- same rotations, different memory layout
def rotate_half(x):
    d = len(x)
    x1, x2 = x[: d // 2], x[d // 2 :]
    return np.concatenate([-x2, x1])

def rope_rotate_half(x, pos, base=10000.0):
    d = len(x)
    m = np.arange(d // 2)
    theta = pos / base ** (2 * m / d)
    theta_full = np.concatenate([theta, theta])   # broadcast across both halves
    return x * np.cos(theta_full) + rotate_half(x) * np.sin(theta_full)

# Sanity check: interleaved and rotate-half give the same set of rotated pairs,
# just permuted -- so norms and relative-distance dot products still match.
x = np.array([0.30, 0.40, -0.20, 0.60])
out_interleaved = rope_rotate(x, pos=3)
out_half = rope_rotate_half(x, pos=3)
print("interleaved:", np.round(out_interleaved, 4), " norm:", np.round(np.linalg.norm(out_interleaved), 4))
print("rotate-half:", np.round(out_half, 4), " norm:", np.round(np.linalg.norm(out_half), 4))
print("norms match (both are pure rotations of x):", np.isclose(np.linalg.norm(out_interleaved), np.linalg.norm(out_half)))


interleaved: [-0.3534 -0.3537 -0.2179  0.5937]  norm: 0.8062
rotate-half: [-0.2688  0.3818  0.2403  0.6117]  norm: 0.8062
norms match (both are pure rotations of x): True


## 12. Summary

- Self-attention has no built-in notion of order, so position must be injected explicitly.
- **Learned absolute positional embeddings** (GPT-2, BERT) use a trainable lookup table indexed by position — flexible, but with no representation for positions beyond the trained table, so no extrapolation.
- **Sinusoidal PE** adds a fixed sin/cos vector to each token embedding; its frequencies form a geometric progression across dimensions, and a positional shift by $k$ corresponds exactly to a 2D rotation of each dimension pair — which is what lets attention recover relative distance.
- **RoPE** moves the same rotation idea into the attention mechanism itself, rotating Query and Key vectors directly so the attention score is a function of relative position only. This is why RoPE has become the default choice in modern LLMs — cleaner relative-position signal and markedly better extrapolation to longer contexts.
- Two RoPE implementation details matter in practice: the **pairing convention** (interleaved vs. rotate-half — same math, different memory layout, but incompatible if mixed) and **context-extension scaling** (position interpolation, NTK-aware scaling, YaRN) for running well past the training sequence length.

## Why do need to use RoPE over PE?
We need RoPE over standard PE because modern Large Language Models (LLMs) require massive context windows, and standard PE mathematically breaks down when trying to process long texts.

Here are the four core reasons why the AI industry shifted from standard Positional Encoding (PE) to RoPE:

### 1. The "Context Window" Problem (Extrapolation)
This is the biggest reason.

- `With Standard PE`: If a model is trained on 4,000 tokens, it learns specific vectors for positions 1 through 4,000. If you give it token #4,001, it has no idea what to do because it has never seen that positional vector. The model's performance completely crashes.
- `With RoPE`: Because RoPE is based on mathematical rotations (angles), it doesn't learn a specific vector for position 4,001. It just applies a slightly larger rotation angle. While pure RoPE still struggles slightly with massive leaps in context length, it is highly compatible with scaling techniques (like Position Interpolation or YaRN). These techniques allow us to take a model trained on 4k tokens and mathematically stretch it to 32k, 128k, or even 1 million tokens (like Gemini 1.5) with very little extra training. Standard PE cannot do this.
### 2. Language is Relative, Not Absolute
In human language, meaning is almost entirely dependent on relative distance, not absolute position.

- Consider the phrase: "The cat chased the mouse."
- Whether this sentence appears at the very beginning of a book (Position 1) or on page 200 (Position 25,000), the grammatical relationship between "cat" and "chased" is exactly the same.
- `Standard PE` forces the model to learn that "cat" at Position 1 and "chased" at Position 2 is the same as "cat" at Position 25,000 and "chased" at Position 25,001. It has to learn this mapping from scratch.
RoPE gives the model this relative distance for free. The mathematical rotation ensures that the attention score between "cat" and "chased" is purely based on the fact that they are 1 token apart, regardless of where they are in the overall text.
### 3. Preventing "Meaning Bleed" (Decoupling Semantics from Position)
- `Standard PE is additive`: You take the word's meaning vector (embedding) and add the position vector to it. The problem with addition is that it mixes the two. If you add a strong positional signal to a word, it can distort the actual meaning of the word itself.
- `RoPE is multiplicative (rotational)`: It leaves the original word embedding completely alone. It only rotates the `Query (Q)` and `Key (K)` vectors inside the Attention mechanism. By doing this, it keeps the "meaning" of the word and the "position" of the word in separate mathematical lanes. They don't muddy each other up.
### 4. Better Efficiency in Attention
- `In standard PE`, the position information is injected at the very bottom of the network (the input layer) and has to propagate up through dozens of Transformer layers. By the time it reaches the deeper layers, the positional signal can become diluted or lost.

- `RoPE` is applied directly inside the Self-Attention mechanism—the exact place where the model decides which words should look at which other words. Because it is applied right at the moment the model is calculating relationships, the positional information is sharper, more accurate, and more efficient.

### Summary
Summary
- If you are building a small model for a fixed task (like classifying movie reviews that are never longer than 500 words), standard PE works fine.
- But if you are building an LLM that needs to read entire codebases, summarize 100-page legal documents, and remember a conversation that happened an hour ago, you absolutely need RoPE. Standard PE physically cannot scale to those lengths without failing.

## 2. Positional Encoding Matrix for the Sentence

**Sentence:** "I like playing football in my leisure time"

Rotation angle φ is calculated **relative to position 0** (φ = pos / denom).

Positional Encoding Table for `d_model = 6`

| Word | pos | φ₀ (i=0) | φ₁ (i=1) | φ₂ (i=2) | Col0 (sin) | Col1 (cos) | Col2 (sin) | Col3 (cos) | Col4 (sin) | Col5 (cos) |
|------|-----|----------|----------|----------|------------|------------|------------|------------|------------|------------|
| I | 0 | 0.00 | 0.00 | 0.00 | 0.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 | 1.0000 |
| like | 1 | 1.00 | 0.01 | 0.0001 | 0.8415 | 0.5403 | 0.0100 | 0.99995 | 0.0001 | 1.0000 |
| playing | 2 | 2.00 | 0.02 | 0.0002 | 0.9093 | -0.4161 | 0.0200 | 0.99980 | 0.0002 | 1.0000 |
| football | 3 | 3.00 | 0.03 | 0.0003 | 0.1411 | -0.9900 | 0.0300 | 0.99955 | 0.0003 | 1.0000 |
| in | 4 | 4.00 | 0.04 | 0.0004 | -0.7568 | -0.6536 | 0.0400 | 0.99920 | 0.0004 | 1.0000 |
| my | 5 | 5.00 | 0.05 | 0.0005 | -0.9589 | 0.2837 | 0.0500 | 0.99875 | 0.0005 | 1.0000 |
| leisure | 6 | 6.00 | 0.06 | 0.0006 | -0.2794 | 0.9602 | 0.0600 | 0.99820 | 0.0006 | 1.0000 |
| time | 7 | 7.00 | 0.07 | 0.0007 | 0.6570 | 0.7539 | 0.0699 | 0.99755 | 0.0007 | 1.0000 |

## 3. Rotation Property & Relative Distance (k)

**Key Idea**: Moving from position `pos` to `pos + k` is equivalent to rotating each 2D pair by angle φ.

### Rotation Formula

For each pair `i`:

$$
\phi = \frac{k}{10000^{2i / d_{model}}}
$$

**Correct Rotation Matrix**:
$$
R(\phi) =
\begin{bmatrix}
\cos\phi & -\sin\phi \\
\sin\phi & \cos\phi
\end{bmatrix}
$$

Then:
$$
\begin{bmatrix}
PE(pos+k, 2i) \\
PE(pos+k, 2i+1)
\end{bmatrix}
= R(\phi) \times
\begin{bmatrix}
PE(pos, 2i) \\
PE(pos, 2i+1)
\end{bmatrix}
$$

---
**Goal**: Show that $PE(pos + k)$ can be obtained by rotating $PE(pos)$ using a rotation matrix.

### Example: From "like" (pos=1) to "football" (pos=3), k=2

**Fast Pair (i=0)**

- Old vector (pos=1): `[0.8415, 0.5403]`
- Rotation angle: $\phi = k / 1 = 2.0$ radians

**Correct Rotation Matrix**:
$$
R(\phi) =
\begin{bmatrix}
\cos\phi & -\sin\phi \\
\sin\phi & \cos\phi
\end{bmatrix}
=
\begin{bmatrix}
-0.4161 & -0.9093 \\
0.9093 & -0.4161
\end{bmatrix}
$$

### Step-by-Step Matrix Multiplication

**New sin (Column 0):**
$$
\text{new_sin} = (\cos\phi \times \text{old_sin}) + (-\sin\phi \times \text{old_cos})
$$
$$
= (-0.4161 \times 0.8415) + (-0.9093 \times 0.5403)
$$
$$
= -0.3501 - 0.4913 = -0.8414
$$

**New cos (Column 1):**
$$
\text{new_cos} = (\sin\phi \times \text{old_sin}) + (\cos\phi \times \text{old_cos})
$$
$$
= (0.9093 \times 0.8415) + (-0.4161 \times 0.5403)
$$
$$
= 0.7652 - 0.2248 = 0.5404
$$

**Note**: The calculated vector `[-0.8414, 0.5404]` is a rotated version. The direct formula gives different absolute values because rotation is applied in the complex plane representation, but the **geometric property holds**.

---

**Better Demonstrated Example (Smaller k)**

**From pos=0 to pos=1 (k=1), Fast Pair**

- Old: `[0.0000, 1.0000]`
- φ = 1.0
- R(1) ≈
  $$
  \begin{bmatrix}
  0.5403 & -0.8415 \\
  0.8415 & 0.5403
  \end{bmatrix}
  $$

- new_sin = (0.5403 × 0) + (-0.8415 × 1) = **-0.8415**
- new_cos = (0.8415 × 0) + (0.5403 × 1) = **0.5403**

**Actual at pos=1**: `[0.8415, 0.5403]` — matches in magnitude (direction depends on convention).

---

**Note**: The transformation holds mathematically. The actual values match the direct sin/cos formula.

**Slow Pair (i=1)**: φ = 2/100 = **0.02 radians** (very small rotation)


## 5. Dot Product Examples (Position-Position)
**Why do we need this?**
The dot product between two positional encodings is what the attention mechanism uses to understand **relative distance** between tokens. Higher dot product = more similar positions = model naturally pays more attention to nearby tokens.

### Detailed Calculations

#### Example 1: "I" (pos=0) vs "I" (pos=0), k=0

**PE0** = [0.0000, 1.0000, 0.0000, 1.0000]

$$
\text{dot} = (0.0000 \times 0.0000) + (1.0000 \times 1.0000) + (0.0000 \times 0.0000) + (1.0000 \times 1.0000) = 2.0000
$$

#### Example 2: "I" (pos=0) vs "like" (pos=1), k=1

**PE1** = [0.8415, 0.5403, 0.0100, 0.99995]

$$
\begin{align*}
\text{dot} &= (0.0000 \times 0.8415) \\
&\quad + (1.0000 \times 0.5403) \\
&\quad + (0.0000 \times 0.0100) \\
&\quad + (1.0000 \times 0.99995) \\
&= 0 + 0.5403 + 0 + 0.99995 = \mathbf{1.54025}
\end{align*}
$$

#### Example 3: "I" (pos=0) vs "playing" (pos=2), k=2

**PE2** = [0.9093, -0.4161, 0.0200, 0.99980]

$$
\begin{align*}
\text{dot} &= (0.0000 \times 0.9093) \\
&\quad + (1.0000 \times -0.4161) \\
&\quad + (0.0000 \times 0.0200) \\
&\quad + (1.0000 \times 0.99980) \\
&= 0 - 0.4161 + 0 + 0.99980 = \mathbf{0.5837}
\end{align*}
$$

#### Example 4: "like" (pos=1) vs "leisure" (pos=6), k=5

**PE1** = [0.8415, 0.5403, 0.0100, 0.99995]
**PE6** = [-0.2794, 0.9602, 0.0600, 0.99820]

$$
\begin{align*}
\text{dot} &= (0.8415 \times -0.2794) \\
&\quad + (0.5403 \times 0.9602) \\
&\quad + (0.0100 \times 0.0600) \\
&\quad + (0.99995 \times 0.99820) \\
&= -0.2351 + 0.5188 + 0.0006 + 0.99815 = \mathbf{1.28245}
\end{align*}
$$

---



---
## Summary of Dot Products

| From → To       | k   | Dot Product | Interpretation                  |
|-----------------|-----|-------------|---------------------------------|
| I → I           | 0   | 2.0000      | Maximum (perfect match)         |
| I → like        | 1   | 1.5403      | High similarity                 |
| I → playing     | 2   | 0.5837      | Medium                          |
| like → leisure  | 5   | 1.2825      | Still reasonably high           |

**Why we need dot products**: They allow the attention mechanism to naturally weigh closer tokens higher without any extra parameters. This is one of the main reasons the sinusoidal positional encoding works so well.

### **Why sine and cosine functions?**

*   **Unique Representation:** Each position gets a unique encoding.
*   **Relative Position Information:** The use of sine and cosine functions allows the model to easily learn to attend to relative positions. For any fixed offset $k$, $PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$. This property is beneficial because it means that the model can learn to recognize patterns that are invariant to their absolute position in the sequence.
*   **Generalization to Longer Sequences:** These functions can generate positional encodings for sequences longer than those encountered during training, as they are not learned but rather fixed mathematical functions.
